# Cuaderno 6: simulación con el proceso completo de PE3 (respuesta a la revisión de la Dra. Gaur)
**Autor:** Aly Gallo · Protocolo ChroniCare-AI v0.5.1 · Continúa los cuadernos 1 a 5.

**Qué hace este cuaderno.** Genera pacientes falsos (la verdad es conocida) y les aplica el análisis completo del protocolo:
nadir robusto → recuperación R<sub>i</sub> → recuperación esperada con *cross-fitting* por paciente (sin BSI basal) → AC<sub>i</sub> = R<sub>i</sub> − Ê[R|S,X] → pendiente promedio β<sub>mean</sub> (una cola) con error estándar por conglomerado (paciente).
Responde a los puntos de la revisión sobre la prueba principal:
- **Mismo resultado y mismo proceso que el primario** (antes se usaba una pendiente simple sobre la fracción).
- **≥ 1,000 repeticiones nulas** (aquí 2,000 para calibrar y otras 1,000 independientes para verificar el tamaño de la prueba), con el **error de Monte Carlo** de cada tamaño y poder.
- **Malla de calibración** de acoplamiento (0 a 0,36) × ruido (0,15; 0,30; 0,60). El valor crítico es el **mayor** percentil 95 de toda la malla, para cada N.
- **Pendiente nula esperada** al lado de cada estimación (el sesgo que producen el acoplamiento y el ruido).
- **Poder de las dos pruebas** (promedio y contraste del techo) con el mismo proceso.
- **Criterios de suficiencia** (poder calibrado ≥ 0,80 y semiancho del IC ≤ efecto de referencia).

**Importante:** todo es simulado. El modelo generador (ruido, patrón de mediciones, forma de la recuperación) son **supuestos** hasta tener el diccionario de datos del hospital; los números definitivos se recalculan con ruido, frecuencia y acoplamiento reales, estimados sin mirar el desenlace.

## Caja de reproducibilidad

In [1]:
import sys, os, platform, datetime, json, time, hashlib, warnings
import numpy as np, pandas as pd
import statsmodels, scipy, joblib, matplotlib
warnings.simplefilter("ignore")

SEED  = 20261113        # semilla maestra; cada celda de la malla usa una semilla derivada y fija (reproducible)
QUICK = False           # True = prueba rápida (unos minutos); False = corrida completa

print("Fecha:", datetime.date.today(), "| Python", sys.version.split()[0], "|", platform.platform())
VERS = {"numpy": np.__version__, "pandas": pd.__version__, "statsmodels": statsmodels.__version__,
        "scipy": scipy.__version__, "joblib": joblib.__version__, "matplotlib": matplotlib.__version__}
print(VERS)
print("SEED =", SEED, "| QUICK =", QUICK, "| núcleos:", os.cpu_count())
print("DATOS: ninguno real. Todo es simulado. Los parámetros del generador vienen de los cuadernos 4 y 5 (cohorte VitalDB, n = 738;")
print("       acoplamiento estimado 0,30 a 0,34, límite superior 0,36; ruido 0,15 y 0,32).")

Fecha: 2026-11-14 | Python 3.13.18 | Linux-6.6.159+-x86_64-with-glibc2.39
{'numpy': '2.4.6', 'pandas': '2.2.3', 'statsmodels': '0.15.0', 'scipy': '1.16.3', 'joblib': '1.6.0', 'matplotlib': '3.10.0'}
SEED = 20261113 | QUICK = False | núcleos: 2
DATOS: ninguno real. Todo es simulado. Los parámetros del generador vienen de los cuadernos 4 y 5 (cohorte VitalDB, n = 738;
       acoplamiento estimado 0,30 a 0,34, límite superior 0,36; ruido 0,15 y 0,32).


## Paso 1. El simulador (se guarda como `fullsim.py`)
Qué hay dentro, en palabras simples:
- Cada paciente tiene un **estado basal verdadero** (BSI) y de 1 a 3 eventos que lo comparten.
- Cada evento hace caer el BSI y luego se recupera. La **caída es más profunda cuando el BSI basal es mayor** en proporción al **acoplamiento** `c` (0 = nada).
- Se mide con **ruido** (`noise`). La línea base es el promedio de 1 o más mediciones previas.
- Nadir = media de las 2 mediciones más bajas en los días 0–4. Valor final = media de las 2 últimas en los días 5–7.
- **Inclusión clínica (primaria):** entran todos los eventos con mediciones suficientes, sin umbral basado en el BSI (como pide la sección 6.4.1). `Rb` repite el análisis con el umbral de caída observada > 0,5 SD de los cuadernos 4 y 5 (sensibilidad). `F` es la fracción recuperada R/D (candidata de la regla de cambio).
- Mundos: **nulo** (sin asociación), **lineal** (+0,05 o +0,10 de fracción recuperable por cada SD de BSI) y **techo** (peor estado basal, menor fracción máxima recuperable).

In [2]:
%%writefile fullsim.py
"""Simulacion 6 (ChroniCare-AI, protocolo v0.5.1): pipeline COMPLETO de PE3 sobre pacientes simulados.
Todo es simulado: no se usa ningun dato real. Los parametros salen de los cuadernos 4 y 5 (VitalDB).
BSI en unidades de SD (mayor = mejor). Pipeline por evento:
  nadir robusto (media de las 2 menores en la ventana de nadir) -> R_i = fin - nadir ->
  recuperacion esperada con cross-fitting POR PACIENTE (sin BSI basal) -> AC_i = R_i - E^[R|S,X] ->
  prueba principal: pendiente de minimos cuadrados beta_mean de AC sobre BSI basal (+covariables), una cola.
Se corre igual con F_i = R_i / D_i (fraccion de la caida recuperada), para la regla de cambio (6.4.7).
"""
import warnings
import numpy as np
from statsmodels.regression.quantile_regression import QuantReg

warnings.filterwarnings('ignore')

# ---- parametros fijos del generador (iguales a los cuadernos 4 y 5 salvo lo indicado) ----
W = 7.0          # ventana de recuperacion (dias)
T0 = 2.0         # dia del nadir verdadero
TAU = 2.0        # escala de tiempo de la recuperacion
WN = 4.0         # ventana de nadir: dias 0 a WN
WE = 2.0         # ventana de cierre: ultimos WE dias
UMBRAL = 0.5     # caida minima observada (SD del BSI) para incluir el evento
KMAX = 14        # maximo de mediciones post-evento por evento
K_FOLDS = 5      # cross-fitting por paciente


def generar(rng, n, noise, c, world="null", effect=0.0, techo=None, dens=4.0):
    """Genera n pacientes (1 a 3 eventos cada uno, comparten su estado basal latente).
    world: 'null' (sin asociacion), 'linear' (la fraccion recuperable sube `effect` por SD de BSI) o 'ceiling'
    (techo: peor estado basal -> menor fraccion recuperable maxima; `techo` = 0.5 moderado, 1.0 fuerte).
    c = acoplamiento (cuanto mas profunda la caida cuando el BSI basal es alto)."""
    age = rng.normal(size=n)
    theta = -0.3 * age + np.sqrt(1 - 0.09) * rng.normal(size=n)       # estado basal verdadero (SD=1)
    ne = 1 + np.minimum(rng.poisson(0.4, n), 2)                         # eventos por paciente
    pid = np.repeat(np.arange(n), ne); E = len(pid)
    th, agez = theta[pid], age[pid]
    S = rng.uniform(0, 1, E)                                            # gravedad clinica (independiente del BSI)
    D = (0.8 + 0.4 * S) * np.clip(1 + c * th, 0.3, None) * np.exp(0.1 * rng.normal(size=E))
    u = rng.beta(4, 2, E) * (1 - 0.25 * S)                              # la recuperacion esperada depende de S
    if world == "linear":
        r = np.clip(u + effect * th, 0.02, 1.0)
    elif world == "ceiling":
        r = np.minimum(u, np.clip(0.95 + techo * th, 0.05, 1.0))
    else:
        r = np.clip(u, 0.02, 1.0)
    # mediciones post-evento (tiempos parametricos, mas densas en los mas graves)
    npost = np.minimum(3 + rng.poisson(dens * S), KMAX)
    t = np.sort(np.where(np.arange(KMAX)[None, :] < npost[:, None], rng.uniform(0.1, W, (E, KMAX)), np.inf), axis=1)
    valid = np.isfinite(t); tt = np.where(valid, t, 0.0)
    k = 1 - np.exp(-(W - T0) / TAU)
    rec = r[:, None] * (1 - np.exp(-(tt - T0) / TAU)) / k
    q = np.where(tt <= T0, tt / T0, 1 - rec)
    x = th[:, None] - D[:, None] * q
    obs = x + noise * rng.normal(size=x.shape)
    npre = 1 + rng.poisson(1.0, E)
    base = th + noise * rng.normal(size=E) / np.sqrt(npre)             # media de las mediciones previas
    # nadir robusto: media de las dos menores dentro de [0, WN]
    inwin = valid & (t <= WN)
    srt = np.sort(np.where(inwin, obs, np.inf), axis=1)[:, :2]
    n_nad = inwin.sum(1)
    nadir = srt.mean(1)
    # valor final: media de las (hasta) dos ultimas mediciones dentro de [W-WE, W]
    inclo = valid & (t >= W - WE)
    n_clo = inclo.sum(1)
    idx = np.argsort(-np.where(inclo, t, -np.inf), axis=1)[:, :2]
    v2 = np.take_along_axis(obs, idx, 1)
    ok2 = np.take_along_axis(inclo, idx, 1)
    fin = np.where(ok2.sum(1) > 0, (v2 * ok2).sum(1) / np.maximum(ok2.sum(1), 1), np.nan)
    Dobs = base - nadir
    inc = (n_nad >= 2) & (n_clo >= 1)                                   # inclusion clinica: solo disponibilidad de mediciones
    R = fin - nadir
    return dict(pid=pid[inc], base=base[inc], S=S[inc], agez=agez[inc], dens=n_nad[inc].astype(float),
                R=R[inc], D=Dobs[inc])


def _cross_fit(Y, Z, pid):
    """AC = Y - E^[Y|Z] con cross-fitting por paciente (todos los eventos de un paciente caen en el mismo pliegue)."""
    fold = pid % K_FOLDS
    pred = np.empty_like(Y)
    for f in range(K_FOLDS):
        te = fold == f; tr = ~te
        b = np.linalg.lstsq(Z[tr], Y[tr], rcond=None)[0]
        pred[te] = Z[te] @ b
    return Y - pred


def _ols_cluster(X, y, pid):
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    bread = np.linalg.inv(X.T @ X)
    starts = np.r_[0, np.flatnonzero(np.diff(pid)) + 1]
    sc = np.add.reduceat(X * e[:, None], starts, axis=0)
    G = len(starts)
    V = bread @ (sc.T @ sc) @ bread * G / max(G - 1, 1)
    return b, np.sqrt(V[1, 1])


def _un_analisis(d, y, quantile=False):
    pid = d["pid"]; one = np.ones(len(pid))
    Z = np.column_stack([one, d["S"], d["agez"], d["dens"]])           # E^[.|S,X]: SIN BSI basal
    X = np.column_stack([one, d["base"], d["S"], d["agez"], d["dens"]])  # modelo primario: CON BSI basal
    ac = _cross_fit(y, Z, pid)
    b, se = _ols_cluster(X, ac, pid)
    out = [float(b[1]), float(se)]
    if quantile:
        b90 = QuantReg(ac, X).fit(q=0.90, max_iter=1000, p_tol=1e-4).params[1]
        b50 = QuantReg(ac, X).fit(q=0.50, max_iter=1000, p_tol=1e-4).params[1]
        out.append(float(b90 - b50))
    return out


def analizar(d, quantile=False):
    """Pipeline completo sobre un conjunto simulado. Tres analisis:
    R  = recuperacion absoluta, TODOS los eventos con mediciones suficientes (inclusion clinica; es el primario del protocolo);
    Rb = igual pero solo eventos con caida observada > UMBRAL (criterio basado en el BSI, como en los cuadernos 4 y 5; sensibilidad);
    F  = fraccion de la caida recuperada R/D, solo eventos con D > UMBRAL (candidato de la regla de cambio 6.4.7)."""
    out = {"n_pac": int(len(np.unique(d["pid"]))), "n_ev": int(len(d["pid"])), "Dbar": float(d["D"].mean())}
    r = _un_analisis(d, d["R"], quantile)
    out["b_R"], out["se_R"] = r[0], r[1]
    if quantile:
        out["dbeta_R"] = r[2]
    k = d["D"] > UMBRAL
    dk = {key: v[k] for key, v in d.items()}
    out["n_pac_b"] = int(len(np.unique(dk["pid"]))); out["Dbar_b"] = float(dk["D"].mean())
    rb = _un_analisis(dk, dk["R"], False); out["b_Rb"], out["se_Rb"] = rb
    f = _un_analisis(dk, np.clip(dk["R"] / dk["D"], -0.5, 1.5), False); out["b_F"], out["se_F"] = f
    return out


def generar_elegibles(rng, n_el, noise, c, world="null", effect=0.0, techo=None):
    """Genera suficientes candidatos para obtener EXACTAMENTE n_el pacientes elegibles (los primeros n_el,
    que son una muestra aleatoria porque los pacientes se generan en orden aleatorio)."""
    n_cand = int(n_el / 0.55) + 100
    while True:
        d = generar(rng, n_cand, noise, c, world, effect, techo)
        up = np.unique(d["pid"])
        if len(up) >= n_el:
            keep = d["pid"] <= up[n_el - 1]
            return {k: v[keep] for k, v in d.items()}
        n_cand *= 2


def repetir(seed_seq, reps, n_el, noise, c, world="null", effect=0.0, techo=None, quantile=False):
    """Repite reps veces (semilla propia, reproducible). n_el = pacientes ELEGIBLES (despues de la regla de inclusion).
    Devuelve un dict de arrays (una entrada por repeticion)."""
    rng = np.random.default_rng(seed_seq)
    res = [analizar(generar_elegibles(rng, n_el, noise, c, world, effect, techo), quantile) for _ in range(reps)]
    return {k: np.array([r[k] for r in res]) for k in res[0]}

Writing fullsim.py


## Paso 2. Configuración y prueba de que es reproducible

In [3]:
if QUICK:
    N_EL = [500, 1000]; NOISES = [0.15, 0.60]; CS = [0.0, 0.36]
    REPS_CAL, REPS_VER, REPS_POW = 100, 100, 100
    N_CEIL, NOISES_CEIL, CS_CEIL, REPS_CEIL = [1000], [0.30], [0.36], 40
else:
    N_EL = [500, 738, 1000, 2000, 3000, 5000]      # pacientes ELEGIBLES (después de la regla de inclusión)
    NOISES = [0.15, 0.30, 0.60]                      # ruido de medición (SD del BSI)
    CS = [0.0, 0.12, 0.24, 0.36]                     # acoplamiento (0.36 = límite superior observado en VitalDB)
    REPS_CAL, REPS_VER, REPS_POW = 2000, 1000, 1000  # calibración / verificación del tamaño / poder
    N_CEIL, NOISES_CEIL, CS_CEIL, REPS_CEIL = [738, 2000, 5000], [0.15, 0.30], [0.0, 0.36], 300
EFFECTS = [0.05, 0.10]     # efecto lineal: fracción recuperable por SD de BSI (0.10 = efecto de referencia del protocolo)
TECHOS = [0.5, 1.0]        # techo moderado y fuerte
OUT = ["R", "F", "Rb"]     # R = primario (inclusión clínica); F = fracción; Rb = sensibilidad con umbral BSI
POWER_TARGET = 0.80

import fullsim as FS
# Prueba de determinismo: misma semilla -> resultados idénticos
a = FS.repetir(np.random.SeedSequence(SEED, spawn_key=(9, 9, 9, 9)), 20, 500, 0.30, 0.24)
b = FS.repetir(np.random.SeedSequence(SEED, spawn_key=(9, 9, 9, 9)), 20, 500, 0.30, 0.24)
print("Misma semilla, dos corridas idénticas:", all(np.array_equal(a[k], b[k]) for k in a))
# Sensibilidad a la semilla: el percentil 95 nulo en una celda con 3 semillas distintas (debe ser parecido)
p = [np.percentile(FS.repetir(np.random.SeedSequence(SEED + s, spawn_key=(8, 8, 8, 8)), 300, 738, 0.30, 0.36)["b_R"], 95) for s in (0, 1, 2)]
print("P95 nulo de β_mean (n=738, ruido 0.30, c=0.36) con 3 semillas:", [round(float(x), 4) for x in p])

Misma semilla, dos corridas idénticas: True
P95 nulo de β_mean (n=738, ruido 0.30, c=0.36) con 3 semillas: [0.1059, 0.1037, 0.1085]


## Paso 3. Correr toda la malla
Para cada combinación de N × ruido × acoplamiento se corren: **calibración nula** (2,000), **verificación nula independiente** (1,000) y los mundos con efecto 0,05 y 0,10 (1,000 cada uno). Usa todos los núcleos. En Colab gratuito tarda unos 30 a 60 minutos: déjalo corriendo.

In [4]:
from joblib import Parallel, delayed
def tarea(key, reps, n, noise, c, world, effect, techo, quantile):
    ss = np.random.SeedSequence(SEED, spawn_key=key)
    return key, FS.repetir(ss, reps, n, noise, c, world, effect, techo, quantile)

tasks = []
for i, n in enumerate(N_EL):
    for j, nz in enumerate(NOISES):
        for k, c in enumerate(CS):
            tasks.append(((i, j, k, 0), REPS_CAL, n, nz, c, "null", 0.0, None, False))
            tasks.append(((i, j, k, 1), REPS_VER, n, nz, c, "null", 0.0, None, False))
            for e, eff in enumerate(EFFECTS):
                tasks.append(((i, j, k, 2 + e), REPS_POW, n, nz, c, "linear", eff, None, False))
t0 = time.time()
R_ = dict(Parallel(n_jobs=-1, verbose=2)(delayed(tarea)(*t) for t in tasks))
print("Malla principal lista en", round((time.time() - t0) / 60, 1), "min;", len(tasks), "tareas")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:  4.8min
[Parallel(n_jobs=-1)]: Done 158 tasks      | elapsed: 29.5min


Malla principal lista en 115.8 min; 288 tareas


[Parallel(n_jobs=-1)]: Done 288 out of 288 | elapsed: 115.8min finished


## Paso 4. Valores críticos calibrados y pendiente nula esperada
La pendiente nula esperada **no es cero**: el acoplamiento la empuja hacia arriba y el ruido o la selección la pueden empujar hacia abajo. Por eso el valor crítico sale de la simulación y no de la tabla normal. **Valor crítico (para cada N) = el mayor percentil 95 nulo de toda la malla.**

In [5]:
def p95_se(x, B=200):
    rg = np.random.default_rng(SEED + 99)
    return float(np.std([np.percentile(rg.choice(x, len(x)), 95) for _ in range(B)]))

crit = {}; filas = []
for i, n in enumerate(N_EL):
    for oc in OUT:
        mejor = None
        for j, nz in enumerate(NOISES):
            for k, c in enumerate(CS):
                x = R_[(i, j, k, 0)]["b_" + oc]
                p = float(np.percentile(x, 95))
                filas.append(dict(n=n, ruido=nz, c=c, resultado=oc, pendiente_nula_media=x.mean(), sd=x.std(), p95=p,
                                  elegibles_media=R_[(i, j, k, 0)]["n_pac"].mean() if oc == "R" else R_[(i, j, k, 0)]["n_pac_b"].mean()))
                if mejor is None or p > mejor[0]: mejor = (p, nz, c, j, k)
        crit[(i, oc)] = dict(valor=mejor[0], ruido=mejor[1], c=mejor[2],
                             mc_se=p95_se(R_[(i, mejor[3], mejor[4], 0)]["b_" + oc]))
nulos = pd.DataFrame(filas)
critdf = pd.DataFrame([dict(n=N_EL[i], resultado=oc, valor_critico=v["valor"], celda_que_manda=f"ruido {v['ruido']}, c {v['c']}", error_MC=v["mc_se"])
                       for (i, oc), v in crit.items()])
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 500)
print("VALORES CRÍTICOS (una cola, 5 %) y la celda de la malla que los determina:")
print(critdf.round(4).to_string(index=False))
print("\nPENDIENTE NULA ESPERADA de β_mean (resultado primario R), media por celda (n = %d):" % N_EL[-1])
print(nulos[(nulos.resultado == "R") & (nulos.n == N_EL[-1])].pivot(index="c", columns="ruido", values="pendiente_nula_media").round(3))

VALORES CRÍTICOS (una cola, 5 %) y la celda de la malla que los determina:
   n resultado  valor_critico    celda_que_manda  error_MC
 500         R         0.1145 ruido 0.15, c 0.36    0.0005
 500         F         0.0120  ruido 0.15, c 0.0    0.0007
 500        Rb         0.1082 ruido 0.15, c 0.36    0.0006
 738         R         0.1109 ruido 0.15, c 0.36    0.0004
 738         F         0.0078  ruido 0.15, c 0.0    0.0005
 738        Rb         0.1026 ruido 0.15, c 0.36    0.0006
1000         R         0.1085 ruido 0.15, c 0.36    0.0005
1000         F         0.0055  ruido 0.15, c 0.0    0.0006
1000        Rb         0.1000 ruido 0.15, c 0.36    0.0005
2000         R         0.1042 ruido 0.15, c 0.36    0.0002
2000         F         0.0012  ruido 0.15, c 0.0    0.0002
2000        Rb         0.0939 ruido 0.15, c 0.36    0.0004
3000         R         0.1023 ruido 0.15, c 0.36    0.0002
3000         F        -0.0003  ruido 0.15, c 0.0    0.0002
3000        Rb         0.0916 ruido 0.15

## Paso 5. Verificación del tamaño de la prueba (con 1,000 repeticiones nulas independientes)

In [6]:
filas = []
for i, n in enumerate(N_EL):
    for j, nz in enumerate(NOISES):
        for k, c in enumerate(CS):
            v = R_[(i, j, k, 1)]
            for oc in OUT:
                s = float(np.mean(v["b_" + oc] > crit[(i, oc)]["valor"]))
                ingenua = float(np.mean(v["b_" + oc] / v["se_" + oc] > 1.645))      # prueba sin calibrar (normal, una cola)
                filas.append(dict(n=n, ruido=nz, c=c, resultado=oc, tamano_calibrado=s, error_MC=np.sqrt(s * (1 - s) / REPS_VER), tamano_sin_calibrar=ingenua))
tam = pd.DataFrame(filas)
print("Tamaño de la prueba (objetivo ≤ 0,05). Peor celda por N, para el resultado primario R:")
w = tam[tam.resultado == "R"].sort_values("tamano_calibrado", ascending=False).groupby("n").head(1).sort_values("n")
print(w.round(3).to_string(index=False))
print("\nTamaño SIN calibrar (z > 1,645), máximo por N, resultado R: muestra por qué hace falta calibrar")
print(tam[tam.resultado == "R"].groupby("n").tamano_sin_calibrar.max().round(3).to_string())

Tamaño de la prueba (objetivo ≤ 0,05). Peor celda por N, para el resultado primario R:
   n  ruido    c resultado  tamano_calibrado  error_MC  tamano_sin_calibrar
 500   0.15 0.36         R             0.053     0.007                  1.0
 738   0.15 0.36         R             0.049     0.007                  1.0
1000   0.15 0.36         R             0.055     0.007                  1.0
2000   0.15 0.36         R             0.060     0.008                  1.0
3000   0.15 0.36         R             0.059     0.007                  1.0
5000   0.15 0.36         R             0.064     0.008                  1.0

Tamaño SIN calibrar (z > 1,645), máximo por N, resultado R: muestra por qué hace falta calibrar
n
500     1.0
738     1.0
1000    1.0
2000    1.0
3000    1.0
5000    1.0


## Paso 6. Poder y criterios de suficiencia

In [7]:
filas = []
for i, n in enumerate(N_EL):
    for j, nz in enumerate(NOISES):
        for k, c in enumerate(CS):
            nul = R_[(i, j, k, 0)]
            for oc in OUT:
                Dbar = {"R": nul["Dbar"].mean(), "Rb": nul["Dbar_b"].mean(), "F": 1.0}[oc]
                dref = 0.10 * Dbar                       # efecto de referencia en las unidades del resultado
                semiancho = 1.96 * nul["se_" + oc].mean()
                for e, eff in enumerate(EFFECTS):
                    b = R_[(i, j, k, 2 + e)]["b_" + oc]
                    pw = float(np.mean(b > crit[(i, oc)]["valor"]))
                    filas.append(dict(n=n, ruido=nz, c=c, resultado=oc, efecto_por_SD=eff, poder_calibrado=pw,
                                      error_MC=np.sqrt(pw * (1 - pw) / REPS_POW), pendiente_media_observada=b.mean(),
                                      pendiente_nula_media=nul["b_" + oc].mean(), delta_ref=dref, semiancho_IC=semiancho,
                                      criterio_i_poder=pw >= POWER_TARGET, criterio_ii_precision=semiancho <= dref))
pot = pd.DataFrame(filas)
pot["suficiente"] = pot.criterio_i_poder & pot.criterio_ii_precision
print("PODER CALIBRADO para el efecto de referencia (0,10 por SD), resultado primario R:")
print("(peor celda de la malla / mediana / mejor celda) y cuántas celdas cumplen AMBOS criterios de suficiencia")
res = []
for oc in OUT:
    for n in N_EL:
        g = pot[(pot.resultado == oc) & (pot.n == n) & (pot.efecto_por_SD == 0.10)]
        g5 = pot[(pot.resultado == oc) & (pot.n == n) & (pot.efecto_por_SD == 0.05)]
        res.append(dict(resultado=oc, n=n, peor=g.poder_calibrado.min(), mediana=g.poder_calibrado.median(), mejor=g.poder_calibrado.max(),
                        celdas_suficientes=f"{int(g.suficiente.sum())}/{len(g)}", peor_con_efecto_0_05=g5.poder_calibrado.min()))
resumen = pd.DataFrame(res)
print(resumen.round(2).to_string(index=False))
print("\nError de Monte Carlo del poder: como máximo ±%.3f (con %d repeticiones)." % (1.96 * 0.5 / np.sqrt(REPS_POW), REPS_POW))

PODER CALIBRADO para el efecto de referencia (0,10 por SD), resultado primario R:
(peor celda de la malla / mediana / mejor celda) y cuántas celdas cumplen AMBOS criterios de suficiencia
resultado    n  peor  mediana  mejor celdas_suficientes  peor_con_efecto_0_05
        R  500  0.00     0.31    1.0               4/12                   0.0
        R  738  0.00     0.36    1.0               4/12                   0.0
        R 1000  0.00     0.38    1.0               5/12                   0.0
        R 2000  0.00     0.56    1.0               5/12                   0.0
        R 3000  0.00     0.67    1.0               5/12                   0.0
        R 5000  0.00     0.80    1.0               6/12                   0.0
        F  500  0.01     0.95    1.0               7/12                   0.0
        F  738  0.01     0.99    1.0               8/12                   0.0
        F 1000  0.00     1.00    1.0               8/12                   0.0
        F 2000  0.00     1.00    

## Paso 6b. Malla acotada por la estimación empírica (escenario realista)
La malla completa es el **peor caso**: incluye acoplamiento 0 y ruido alto, que dan un valor crítico muy exigente. El protocolo dice que el acoplamiento y el ruido se **estiman sin mirar el desenlace** con los valores previos y el nadir del hospital. Aquí se repite el cálculo con una malla acotada alrededor de lo medido en VitalDB (acoplamiento 0,24 a 0,36; ruido 0,15 a 0,30) para ver cuánto poder se recupera **si** esa estimación se confirma con datos del hospital.

In [8]:
C_SUB = [c for c in CS if c >= 0.24]; NZ_SUB = [z for z in NOISES if z <= 0.30]
filas = []
for i, n in enumerate(N_EL):
    for oc in OUT:
        cs_ = max(np.percentile(R_[(i, j, k, 0)]["b_" + oc], 95) for j, z in enumerate(NOISES) if z in NZ_SUB for k, c in enumerate(CS) if c in C_SUB)
        for e, eff in enumerate(EFFECTS):
            pw = [float(np.mean(R_[(i, j, k, 2 + e)]["b_" + oc] > cs_)) for j, z in enumerate(NOISES) if z in NZ_SUB for k, c in enumerate(CS) if c in C_SUB]
            sz = [float(np.mean(R_[(i, j, k, 1)]["b_" + oc] > cs_)) for j, z in enumerate(NOISES) if z in NZ_SUB for k, c in enumerate(CS) if c in C_SUB]
            filas.append(dict(resultado=oc, n=n, efecto_por_SD=eff, valor_critico_acotado=cs_, poder_peor=min(pw), poder_mediano=float(np.median(pw)),
                              tamano_max=max(sz), error_MC_poder=1.96 * 0.5 / np.sqrt(REPS_POW)))
acot = pd.DataFrame(filas)
print("MALLA ACOTADA: acoplamiento", C_SUB, "ruido", NZ_SUB)
print(acot[acot.efecto_por_SD == 0.10].round(3).to_string(index=False))
print("\nCon efecto 0,05 por SD:")
print(acot[acot.efecto_por_SD == 0.05].round(3).to_string(index=False))

MALLA ACOTADA: acoplamiento [0.24, 0.36] ruido [0.15, 0.3]
resultado    n  efecto_por_SD  valor_critico_acotado  poder_peor  poder_mediano  tamano_max  error_MC_poder
        R  500            0.1                  0.114       0.805          0.986       0.053           0.031
        F  500            0.1                 -0.001       0.920          0.989       0.037           0.031
       Rb  500            0.1                  0.108       0.618          0.973       0.049           0.031
        R  738            0.1                  0.111       0.898          1.000       0.049           0.031
        F  738            0.1                 -0.006       0.984          0.998       0.056           0.031
       Rb  738            0.1                  0.103       0.769          0.996       0.049           0.031
        R 1000            0.1                  0.109       0.957          1.000       0.055           0.031
        F 1000            0.1                 -0.009       0.995          1.0

## Paso 7. Contraste del techo (Δβ, secundario) con el mismo proceso
Se calibra con su propio nulo (percentil 95; el mayor de la malla reducida) y se compara con la prueba promedio **en los mismos mundos con techo**. La regresión cuantílica es lenta, por eso aquí hay menos repeticiones y una malla más pequeña; el error de Monte Carlo se muestra.

In [9]:
tasks = []
for i, n in enumerate(N_CEIL):
    for j, nz in enumerate(NOISES_CEIL):
        for k, c in enumerate(CS_CEIL):
            tasks.append(((100 + i, j, k, 4), REPS_CEIL, n, nz, c, "null", 0.0, None, True))
            for e, tc in enumerate(TECHOS):
                tasks.append(((100 + i, j, k, 5 + e), REPS_CEIL, n, nz, c, "ceiling", 0.0, tc, True))
t0 = time.time()
RC = dict(Parallel(n_jobs=-1, verbose=2)(delayed(tarea)(*t) for t in tasks))
print("Contraste del techo listo en", round((time.time() - t0) / 60, 1), "min")
filas = []
for i, n in enumerate(N_CEIL):
    cdb = max(np.percentile(RC[(100 + i, j, k, 4)]["dbeta_R"], 95) for j in range(len(NOISES_CEIL)) for k in range(len(CS_CEIL)))
    cm = crit[(N_EL.index(n), "R")]["valor"]
    for j, nz in enumerate(NOISES_CEIL):
        for k, c in enumerate(CS_CEIL):
            for e, tc in enumerate(TECHOS):
                r = RC[(100 + i, j, k, 5 + e)]
                pd_ = float(np.mean(r["dbeta_R"] > cdb)); pm = float(np.mean(r["b_R"] > cm))
                filas.append(dict(n=n, ruido=nz, c=c, techo=tc, poder_techo_dbeta=pd_, error_MC_dbeta=np.sqrt(pd_ * (1 - pd_) / REPS_CEIL),
                                  poder_prueba_promedio=pm, error_MC_promedio=np.sqrt(pm * (1 - pm) / REPS_CEIL), valor_critico_dbeta=cdb))
techo = pd.DataFrame(filas)
print(techo.round(3).to_string(index=False))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.


Contraste del techo listo en 9.4 min
   n  ruido    c  techo  poder_techo_dbeta  error_MC_dbeta  poder_prueba_promedio  error_MC_promedio  valor_critico_dbeta
 738   0.15 0.00    0.5              0.000           0.000                  0.000              0.000                0.071
 738   0.15 0.00    1.0              0.000           0.000                  0.873              0.019                0.071
 738   0.15 0.36    0.5              0.077           0.015                  0.940              0.014                0.071
 738   0.15 0.36    1.0              0.027           0.009                  1.000              0.000                0.071
 738   0.30 0.00    0.5              0.000           0.000                  0.000              0.000                0.071
 738   0.30 0.00    1.0              0.000           0.000                  0.663              0.027                0.071
 738   0.30 0.36    0.5              0.050           0.013                  0.597              0.028         

[Parallel(n_jobs=-1)]: Done  36 out of  36 | elapsed:  9.4min finished


## Paso 8. Figura, archivos y huellas

In [10]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for eff, st in ((0.10, "-o"), (0.05, "--s")):
    g = pot[(pot.resultado == "R") & (pot.efecto_por_SD == eff)].groupby("n").poder_calibrado.agg(["min", "median"])
    ax[0].plot(g.index, g["min"], st, label=f"peor celda, efecto {eff}/SD"); ax[0].plot(g.index, g["median"], st, alpha=.4, label=f"mediana, efecto {eff}/SD")
ax[0].axhline(0.8, color="k", lw=.8); ax[0].set_xscale("log"); ax[0].set_xlabel("pacientes elegibles"); ax[0].set_ylabel("poder calibrado")
ax[0].set_title("Prueba principal β_mean (resultado R)"); ax[0].legend(fontsize=7)
g = nulos[(nulos.resultado == "R") & (nulos.n == N_EL[-1])]
for nz, gg in g.groupby("ruido"): ax[1].plot(gg.c, gg.pendiente_nula_media, "-o", label=f"ruido {nz}")
ax[1].axhline(0, color="k", lw=.8); ax[1].set_xlabel("acoplamiento c"); ax[1].set_ylabel("pendiente nula esperada")
ax[1].set_title("Sesgo de β_mean cuando NO hay efecto"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig("sim6_figura.png", dpi=150); plt.close()

nulos.to_csv("sim6_nulo_por_celda.csv", index=False); critdf.to_csv("sim6_valores_criticos.csv", index=False)
tam.to_csv("sim6_tamano_verificacion.csv", index=False); pot.to_csv("sim6_poder_suficiencia.csv", index=False)
resumen.to_csv("sim6_resumen_por_n.csv", index=False); acot.to_csv("sim6_malla_acotada.csv", index=False); techo.to_csv("sim6_techo_dbeta.csv", index=False)
json.dump(dict(fecha=str(datetime.date.today()), semilla=SEED, quick=QUICK, versiones=VERS, N_elegibles=N_EL, ruidos=NOISES, acoplamientos=CS,
               reps=dict(calibracion=REPS_CAL, verificacion=REPS_VER, poder=REPS_POW, techo=REPS_CEIL), efectos=EFFECTS, techos=TECHOS,
               ceiling_grid=dict(N=N_CEIL, ruidos=NOISES_CEIL, c=CS_CEIL), datos_reales="ninguno (todo simulado)"), open("sim6_parametros.json", "w"), indent=2, ensure_ascii=False)
open("requirements_sim6.txt", "w").write("\n".join(f"{k}=={v}" for k, v in VERS.items()) + "\n")
arch = ["sim6_nulo_por_celda.csv", "sim6_valores_criticos.csv", "sim6_tamano_verificacion.csv", "sim6_poder_suficiencia.csv",
        "sim6_resumen_por_n.csv", "sim6_malla_acotada.csv", "sim6_techo_dbeta.csv", "sim6_parametros.json", "requirements_sim6.txt", "sim6_figura.png", "fullsim.py"]
man = {a: hashlib.sha256(open(a, "rb").read()).hexdigest()[:16] for a in arch}
json.dump(man, open("sim6_huellas.json", "w"), indent=2)
print("Archivos guardados (sube TODOS a GitHub junto con este cuaderno ya ejecutado):"); [print(" ", a, man.get(a, "")) for a in arch + ["sim6_huellas.json"]]
with __import__("zipfile").ZipFile("sim6_resultados.zip", "w") as z:
    for a in arch + ["sim6_huellas.json"]: z.write(a)
try:
    from google.colab import files; files.download("sim6_resultados.zip")
except Exception as e:
    print("(descarga manual: sim6_resultados.zip en el panel de archivos)")

Archivos guardados (sube TODOS a GitHub junto con este cuaderno ya ejecutado):
  sim6_nulo_por_celda.csv a4a67234e6da95f7
  sim6_valores_criticos.csv ac91c9981c1fa3e4
  sim6_tamano_verificacion.csv 606ac0bc27b0be15
  sim6_poder_suficiencia.csv 5a64ff89ded6f359
  sim6_resumen_por_n.csv 35a86db74eff50e1
  sim6_malla_acotada.csv 804b4af7eb1bc574
  sim6_techo_dbeta.csv 4fce53178c276678
  sim6_parametros.json daf1ce6e4789b1be
  requirements_sim6.txt ab136d0f9a24c950
  sim6_figura.png 5b79315a4436dc27
  fullsim.py ab952b938f487b29
  sim6_huellas.json 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cómo leer los resultados
- **Valores críticos:** son el umbral que debe superar la pendiente estimada para declararla significativa (una cola, 5 %). Salen del peor caso de la malla, por eso protegen contra un acoplamiento o ruido desfavorables.
- **Pendiente nula esperada:** es lo que la estimación valdría **aunque no hubiera efecto**. En el artículo final se reporta junto a la estimación.
- **Tamaño de la prueba:** debe quedar ≤ 0,05 (con su error de Monte Carlo). La versión sin calibrar muestra cuánto se equivocaría la prueba usual.
- **Poder:** busca ≥ 0,80 en la **peor celda** (lectura conservadora) y mira cuántas celdas cumplen los dos criterios.
- **Criterio ii:** el semiancho del IC debe caber dentro del efecto de referencia.
- **Resultado `F`:** el denominador lleva el error de medición de la línea base, así que su pendiente queda atenuada y sesgada hacia abajo. Eso se compara con `R` para la regla de cambio de la sección 6.4.7.
- **Límites:** es una simulación con supuestos propios del generador; el modelo de caída y recuperación, el ruido, el patrón de mediciones y el número de eventos se recalculan con datos reales (sin mirar el desenlace) antes de registrar.